# Modelado Estocástico
## Clase 4 / Clase 5 - Regresión Múltiple - Notación matricial

En la teoría analizamos el modelo con dos variables explicativas:
$$y_{i} = \alpha + \beta_{1}x_{1i} + \beta_{2}x_{2i} + u_{i}$$

A continuación generalizamos el modelo clásico de Mínimos Cuadrados Ordinarios (MCO) utilizando la notación matricial:  $$\mathbf{y} = \mathbf{X}\boldsymbol{\beta} + \mathbf{u}$$
Donde:

* $\mathbf{y} \in \mathbb{R}^{n \times 1}$ es el vector de observaciones de la variable dependiente.
* $\mathbf{X} \in \mathbb{R}^{n \times k}$ es la matriz de diseño que incluye una primera columna de unos (para el intercepto $\alpha$ o $\beta_0$) y las $k-1$ variables regresoras.  
* $\boldsymbol{\beta} \in \mathbb{R}^{k \times 1}$ es el vector de parámetros a estimar.
* $\mathbf{u} \in \mathbb{R}^{n \times 1}$ es el vector de perturbaciones estocásticas (errores).

El método de MCO busca minimizar la Suma de los Residuos al Cuadrado ($RSS$)  $\mathbf{e}'\mathbf{e}$ :

  $$(\mathbf{y} - \mathbf{X}\boldsymbol{\hat{\beta}})'(\mathbf{y} - \mathbf{X}\boldsymbol{\hat{\beta}})
$$

Derivando respecto a $\boldsymbol{\hat{\beta}}$ e igualando a cero, obtenemos las Ecuaciones Normales en su forma matricial:

  $$\mathbf{X}'\mathbf{X}\boldsymbol{\hat{\beta}} = \mathbf{X}'\mathbf{y}$$
  
  Si la matriz $\mathbf{X}'\mathbf{X}$ es de rango completo (no hay multicolinealidad perfecta) , posee inversa y la solución analítica es única:
  $$\hat{\boldsymbol{\beta}} = (\mathbf{X}'\mathbf{X})^{-1}\mathbf{X}'\mathbf{y}$$

  Si además los supuestos de
homocedasticidad y no autocorrelación se cumplen (Gauss-Markov):

$$Var(\boldsymbol{\hat{\beta}}) = \sigma^2(\mathbf{X}'\mathbf{X})^{-1}$$


**Importante:** Si las variables predictoras están altamente correlacionadas (Multicolinealidad), el determinante de $\mathbf{X}'\mathbf{X}$ tiende a cero. Esto provoca que su inversa explote, inflando la varianza de los parámetros e inestabilizando los algoritmos de optimización por gradiente.

* Importamos los recursos necesarios

In [ ]:
import numpy as np
import statsmodels.api as sm

* Generamos matrices X e Y de ejemplo

In [ ]:
X = np.array([[1, 2], [1, 3], [1, 1], [1, 5], [1, 9]])
y = np.array([4, 7, 3, 9, 17])

In [ ]:
X

array([[1, 2],
       [1, 3],
       [1, 1],
       [1, 5],
       [1, 9]])

### Algunas operaciones sobre matrices:

* `X.transpose()` y `X.T` permiten obtener la matriz transpuesta de X:

In [ ]:
X.transpose()

array([[1, 1, 1, 1, 1],
       [2, 3, 1, 5, 9]])

In [ ]:
X.T


array([[1, 1, 1, 1, 1],
       [2, 3, 1, 5, 9]])

Comparamos las dimensiones de X y su matriz transpuesta:

In [ ]:
print(f"Dimensión de X = {X.shape} \nDimensión de X transpuesta = {X.transpose().shape}")

Dimensión de X = (5, 2) 
Dimensión de X transpuesta = (2, 5)


Ejemplo de la teórica, cálculo manual:

Para el cálculo manual vamos a calcular los estimadores usando las expresiones de matrices.

Sabemos que $\hat{\beta} = (X'X)^{-1}(X'y)$

`np.lininv` calcula la inversa de una matriz y `@` multiplica matrices.


In [ ]:
def estimar_mco_matricial(X: np.ndarray, y: np.ndarray):
    """Calcula el vector de parámetros beta y su matriz de varianza-covarianza."""
    XtX = X.T @ X

    XtX_inv = np.linalg.inv(XtX)

    beta = XtX_inv @ X.T @ y

    n, k = X.shape
    y_pred = X @ beta
    residuos = y - y_pred
    rss = float(residuos.T @ residuos)
    s_cuadrado = rss / (n - k)

    var_beta = s_cuadrado * XtX_inv

    return beta, var_beta

**IMPORTANTE** La inversión de matrices es costosa computacionalmente y, además, puede generar pérdida de precisión. Si el objetivo es la resolución de sistemas de ecuaciones siempre es conveniente preferir un método más estable desde su implementación antes que la fórmula manual, como por ejemplo `numpy.linalg.solve()`.

En el ejemplo se mantuvo la inversa solo a título ilustrativo, al igual que los cálculos manuales. Pero siempre es importante analizar el impacto y la magnitud de los errores que pueden incorporar los criterios de cálculo elegido y priorizar los optimizados y estables provistos por las bibliotecas.

In [ ]:
beta_manual, vcov_manual = estimar_mco_matricial(X, y)

modelo = sm.OLS(y, X).fit()

print(f"Beta manual: {beta_manual} \nStd_Errors (raíz de la diagonal):{np.sqrt(np.diag(vcov_manual))} \n\nBeta de statsmodels: {modelo.params} \nStd_Errors: {modelo.bse}")

Beta manual: [1.   1.75] 
Std_Errors (raíz de la diagonal):[0.54772256 0.1118034 ] 

Beta de statsmodels: [1.   1.75] 
Std_Errors: [0.54772256 0.1118034 ]


### Cálculo del $s^2$ usando matrices:

Si recapitulamos los valores del modelo de la librería, veremos que la segunda columna es el desvío estándar del $\hat{\beta}$ correspondiente a la pendiente que multiplique la $X$ que corresponda (en regresión múltiple) o del intercepeto (fila "const")

 Estos valores son la raíz de los elementos de la diagonal principal de la matriz

$Var(\hat{\beta}) =  s^2 @ (X'X)^{-1}$

In [ ]:
var_beta_sombrero = modelo.scale * (np.linalg.inv(X.T @ X))
print(var_beta_sombrero)

[[ 0.3    -0.05  ]
 [-0.05    0.0125]]


Utilizando el modelo calculado:

In [ ]:
modelo.cov_params()

array([[ 0.3   , -0.05  ],
       [-0.05  ,  0.0125]])

Valor de $\hat{y} = \mathbf{X}\boldsymbol{\hat{\beta}}$

In [ ]:
y_sombrero_manual = X @ beta_manual
y_modelo = modelo.fittedvalues
print(f"Valor predicho de y, calculo manual:{y_sombrero_manual}\n\nValor predicho de y, usando statsmodels: {y_modelo}")

Valor predicho de y, calculo manual:[ 4.5   6.25  2.75  9.75 16.75]

Valor predicho de y, usando statsmodels: [ 4.5   6.25  2.75  9.75 16.75]


Residuos $e = y - \hat{y}$

In [ ]:
residuos_manual = y - y_sombrero_manual
residuos_modelo = modelo.resid
print(f"Residuos manual:{residuos_manual}\n\nResiduos usando statsmodels: {residuos_modelo}")

Residuos manual:[-0.5   0.75  0.25 -0.75  0.25]

Residuos usando statsmodels: [-0.5   0.75  0.25 -0.75  0.25]


### Matrices de proyección

Notar que:

$\hat{y} = X \hat{\beta} = X (X'X)^{-1}(X'y) = Py$

donde
$P = X(X'X)^{-1}X'$

La matriz P es una matriz de $nxn$ idempotente que proyecta sobre el espacio generado por las columnas de X.

Vemos que P satisface la definición de matriz idempotente, ya que $P @ P = P$

In [ ]:
P = X @ np.linalg.inv(X.T @ X) @ X.T
np.allclose(P @ P, P)

True

Tenemos otra matriz idempotente:

$M = I - P$

¿De dónde viene? De los residuos:

$e = y - \hat{y} = y - P @ y = (I-P) @ y = M @ y$

Donde la matriz identidad I tiene la misma dimension que P. Para generar la matriz identidad utilizamos `np.eye(n)`.

Notar que M es idempotente: $M @ M = M$   
($M$ es de $nxn$)

$M = I - P = I - X (X'X)^{-1}X'$

In [ ]:
M = np.eye(P.shape[0]) - P

e_ = M @ y
np.allclose(e_, modelo.resid)

True